# SentinelAI: MuRIL Hinglish Fine-tuning

**Run this notebook with a Google Colab GPU runtime.**

This notebook fine-tunes `google/muril-base-cased` on the prepared Hinglish sentiment dataset.

## 1. Environment Check & Dependency Installation

In [8]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os

print(os.listdir("/content/drive"))

print(os.listdir("/content/drive/MyDrive/Sentinel AI/data/processed/hinglish"))

['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']
['validation.tsv', 'train.tsv']


In [11]:
import torch
import transformers
import sys

print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
    print("CUDA availability: True")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # get memory in GB
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {mem:.2f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("MPS availability: True")
else:
    print("CUDA availability: False")

print(f"\nUsing device: {device}")

Python version: 3.12.12
PyTorch version: 2.9.0+cu126
Transformers version: 4.57.6
CUDA availability: True
GPU name: Tesla T4
GPU memory: 15.64 GB

Using device: cuda


## 2. Configuration

In [12]:
import os

# Path configuration
# You can change this to point to Google Drive if mounted, e.g. '/content/drive/MyDrive/sentinelai/data/processed/hinglish'
DATA_DIR = "/content/drive/MyDrive/Sentinel AI/data/processed/hinglish"

TRAIN_PATH = os.path.join(DATA_DIR, "train.tsv")
VALID_PATH = os.path.join(DATA_DIR, "validation.tsv")

MODEL_NAME = "google/muril-base-cased"
OUTPUT_DIR = "/content/drive/MyDrive/Sentinel AI/models/hinglish_muril"
REPORTS_DIR = "/content/drive/MyDrive/Sentinel AI/reports"

# Training configuration
MAX_LENGTH = 128
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Dataset Loading & Validation

In [13]:
import pandas as pd

def load_data(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Make sure to upload the dataset or mount Google Drive.")
    
    df = pd.read_csv(path, sep='\t', header=None, names=['id', 'text', 'label'])
    # Basic validation
    df = df.dropna(subset=['text', 'label'])
    df['label'] = df['label'].astype(int)
    return df

train_df = load_data(TRAIN_PATH)
valid_df = load_data(VALID_PATH)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(valid_df)}")

# Display label distribution
print("\nTrain Distribution:\n", train_df['label'].value_counts())
print("\nValidation Distribution:\n", valid_df['label'].value_counts())

Train size: 14091
Validation size: 1567

Train Distribution:
 label
1    5310
2    4698
0    4083
Name: count, dtype: int64

Validation Distribution:
 label
1    591
2    522
0    454
Name: count, dtype: int64


## 4. Tokenization

In [14]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/14091 [00:00<?, ? examples/s]

Map:   0%|          | 0/1567 [00:00<?, ? examples/s]

## 5. Model Initialization

In [15]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

## 6. Training & Evaluation Setup

In [16]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='macro')
    weighted_f1 = f1_score(labels, predictions, average='weighted')
    macro_precision = precision_score(labels, predictions, average='macro')
    macro_recall = recall_score(labels, predictions, average='macro')
    
    per_class_f1 = f1_score(labels, predictions, average=None)
    per_class_precision = precision_score(labels, predictions, average=None)
    per_class_recall = recall_score(labels, predictions, average=None)
    
    cm = confusion_matrix(labels, predictions)
    
    # Returning standard metrics for logging
    # Trainer expects scalar values in this dictionary.
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "f1_0": per_class_f1[0],
        "f1_1": per_class_f1[1],
        "f1_2": per_class_f1[2],
        "prec_0": per_class_precision[0],
        "prec_1": per_class_precision[1],
        "prec_2": per_class_precision[2],
        "rec_0": per_class_recall[0],
        "rec_1": per_class_recall[1],
        "rec_2": per_class_recall[2],
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    fp16=torch.cuda.is_available() and torch.cuda.is_bf16_supported() == False, # Use fp16 if CUDA is available but bf16 is not
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(), # Use bf16 if supported
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-2873217103.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 7. Training

In [17]:
import time
start_time = time.time()
train_result = trainer.train()
train_time = time.time() - start_time
print(f"Training took {train_time:.2f} seconds")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,F1 0,F1 1,F1 2,Prec 0,Prec 1,Prec 2,Rec 0,Rec 1,Rec 2
1,1.036300,0.883994,0.590300,0.578762,0.568928,0.584491,0.605704,0.632231,0.408120,0.695935,0.595331,0.553623,0.604520,0.674009,0.323181,0.819923
2,0.813700,0.806376,0.636886,0.641816,0.636827,0.641507,0.642160,0.666667,0.553011,0.705769,0.661605,0.554422,0.708494,0.671806,0.551607,0.703065
3,0.726800,0.804253,0.648373,0.653483,0.648106,0.652706,0.654779,0.691649,0.568995,0.699805,0.672917,0.572899,0.712302,0.711454,0.565144,0.687739


Training took 1546.19 seconds


## 8. Best Model Saving

In [18]:
trainer.save_model(OUTPUT_DIR)
print(f"Best model saved to {OUTPUT_DIR}")

Best model saved to /content/drive/MyDrive/Sentinel AI/models/hinglish_muril


## 9. Metrics & Report Generation

In [19]:
import json
from datetime import datetime

# Evaluate on validation set one last time to get the final confusion matrix and metrics directly
eval_preds = trainer.predict(tokenized_valid)
predictions = np.argmax(eval_preds.predictions, axis=-1)
labels = eval_preds.label_ids

metrics = eval_preds.metrics
cm = confusion_matrix(labels, predictions).tolist()

report = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "environment": {
        "python_version": sys.version,
        "pytorch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    },
    "model": {
        "base_model": MODEL_NAME
    },
    "dataset": {
        "train_size": len(train_dataset),
        "validation_size": len(valid_dataset)
    },
    "configuration": {
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "seed": SEED,
        "fp16": training_args.fp16,
        "bf16": training_args.bf16
    },
    "results": {
        "accuracy": metrics["test_accuracy"],
        "macro_f1": metrics["test_macro_f1"],
        "weighted_f1": metrics["test_weighted_f1"],
        "per_class": {
            "0": {
                "precision": metrics["test_prec_0"],
                "recall": metrics["test_rec_0"],
                "f1": metrics["test_f1_0"]
            },
            "1": {
                "precision": metrics["test_prec_1"],
                "recall": metrics["test_rec_1"],
                "f1": metrics["test_f1_1"]
            },
            "2": {
                "precision": metrics["test_prec_2"],
                "recall": metrics["test_rec_2"],
                "f1": metrics["test_f1_2"]
            }
        },
        "confusion_matrix": cm
    },
    "training_time_seconds": train_time
}

with open(os.path.join(REPORTS_DIR, "hinglish_muril_results.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

md_content = f"""# Hinglish MuRIL Fine-tuning Report

**Timestamp:** {report['timestamp']}
**Base Model:** {MODEL_NAME}
**Device:** {report['environment']['device']} ({report['environment']['gpu']})
**Training Time:** {train_time:.2f} seconds

## Configuration
- max_length: {MAX_LENGTH}
- batch_size: {BATCH_SIZE}
- learning_rate: {LEARNING_RATE}
- num_epochs: {NUM_EPOCHS}
- weight_decay: {WEIGHT_DECAY}
- warmup_ratio: {WARMUP_RATIO}
- seed: {SEED}

## Results
- **Macro F1:** {metrics['test_macro_f1']:.4f}
- **Accuracy:** {metrics['test_accuracy']:.4f}
- **Weighted F1:** {metrics['test_weighted_f1']:.4f}

### Comparison
| Model | Accuracy | Macro F1 | Weighted F1 |
|---|---|---|---|
| TF-IDF + Logistic Regression | 0.6605 | 0.6641 | 0.6602 |
| MuRIL | {metrics['test_accuracy']:.4f} | {metrics['test_macro_f1']:.4f} | {metrics['test_weighted_f1']:.4f} |

### Per-Class Metrics
**0 (Negative):** Precision: {metrics['test_prec_0']:.4f} | Recall: {metrics['test_rec_0']:.4f} | F1: {metrics['test_f1_0']:.4f}
**1 (Neutral):** Precision: {metrics['test_prec_1']:.4f} | Recall: {metrics['test_rec_1']:.4f} | F1: {metrics['test_f1_1']:.4f}
**2 (Positive):** Precision: {metrics['test_prec_2']:.4f} | Recall: {metrics['test_rec_2']:.4f} | F1: {metrics['test_f1_2']:.4f}

### Confusion Matrix
```
{cm[0]}
{cm[1]}
{cm[2]}
```
"""

with open(os.path.join(REPORTS_DIR, "hinglish_muril_results.md"), "w", encoding="utf-8") as f:
    f.write(md_content)

print("Reports generated successfully.")

Reports generated successfully.


/tmp/ipython-input-67353036.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


## 10. Inference Smoke Test

In [20]:
from transformers import pipeline

classifier = pipeline("text-classification", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR)

test_texts = [
    "ye product bilkul bekar hai",
    "bahut badhiya service hai inki"
]

print("Smoke Test Predictions:")
for text in test_texts:
    result = classifier(text)[0]
    print(f"'{text}' -> {result['label']} (score: {result['score']:.4f})")

The tokenizer you are loading from '/content/drive/MyDrive/Sentinel AI/models/hinglish_muril' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


Smoke Test Predictions:
'ye product bilkul bekar hai' -> negative (score: 0.5018)
'bahut badhiya service hai inki' -> positive (score: 0.7555)
